In [58]:
from pyiceberg.catalog import load_catalog
import polars as pl
import s3fs
import duckdb

cat = load_catalog("dev", uri="http://rest:8181",
                   **{
                        "s3.endpoint": "http://minio:9000",
                        "s3.access-key-id": "minioadmin",
                        "s3.secret-access-key": "minioadmin",
                   }
)

fs = s3fs.S3FileSystem(
    key=cat.properties["s3.access-key-id"],
    secret=cat.properties["s3.secret-access-key"],
    endpoint_url=cat.properties["s3.endpoint"]
)

In [27]:
cat.properties

{'uri': 'http://rest:8181',
 's3.endpoint': 'http://minio:9000',
 's3.access-key-id': 'minioadmin',
 's3.secret-access-key': 'minioadmin'}

In [71]:
with fs.open("s3://raw/grade_tributaria_paulao/lista de produtos pdf.xlsx", "rb") as f:
    import io
    content = io.BytesIO(f.read())
    original_df = pl.read_excel(content)

df = original_df.rename(str.lower)
df = df.with_columns(
    pl.col("cst_icms").cast(pl.String).alias("cst_icms"),
    pl.col("csosn").cast(pl.String).alias("csosn"),
    cest = pl.col("cest").cast(pl.String),
    pis = pl.col("pis").cast(pl.String),
    cofins = pl.col("cofins").cast(pl.String),
    ipi = pl.col("ipi").cast(pl.String),
)
df = df.with_columns([
    pl.when(pl.col("cest").is_null()).then(pl.lit("5102")).otherwise(pl.lit("5405")).alias("cfop")
])
cols_trib = [
    "cfop",
    # "cest",
    "%fcp",
    "cst_icms",
    "aliq.icms",
    "pis",
    "%pis",
    "cofins",
    "%cof",
    "%red",
    "ipi",
    "%ipi",
]

df_trib = df.select(cols_trib).unique()
df_trib = df_trib.with_row_index("id_trib", offset=1)
df_trib

relacao = (
    df
    .join(df_trib, on=cols_trib, how="left")
    .select(["ncm", "id_trib"])
    .unique()
    .sort("ncm")
)

ncm_cest = (
    df.select(["ncm", "cest"])
    .unique()
    .filter(pl.col("cest").is_not_null())
    .sort("cest")
)

tribprod_values = []
tribpc_values = []
result_grades = []
for row in df_trib.to_dicts():
    descricao = f"Venda {row['cfop']}"
    result_grades.append(f"INSERT INTO gradetributaria (idEmpresa, descricao) VALUES (2, {descricao})")
    

with fs.open("s3://raw/grade_tributaria_paulao/relacao-1.csv", "wb") as f:
    relacao.write_csv(f)
    
with fs.open("s3://raw/grade_tributaria_paulao/tributacoes-1.csv", "wb") as f:
    df_trib.write_csv(f)
    
with fs.open("s3://raw/grade_tributaria_paulao/ncm_cest-1.csv", "wb") as f:
    ncm_cest.write_csv(f)

con = duckdb.connect()

con.register("relacao", relacao)
con.register("tributacoes", df_trib)
con.register("produtos", df)
con.register("ncm_cest", ncm_cest)

resultado = con.sql("""
    SELECT p.cest
    , n.cest as 'NCM_REF'
    , p.*
    FROM produtos p
    JOIN ncm_cest n
        ON p.ncm = n.ncm
    WHERE p.cest != n.cest
""").pl()

display(con.sql("""
    select * from tributacoes
""").pl())

Could not determine dtype for column 3, falling back to string
Could not determine dtype for column 5, falling back to string
Could not determine dtype for column 13, falling back to string
Could not determine dtype for column 15, falling back to string


id_trib,cfop,%fcp,cst_icms,aliq.icms,pis,%pis,cofins,%cof,%red,ipi,%ipi
u32,str,i64,str,f64,str,f64,str,i64,str,str,str
1,"""5102""",null,"""40""",null,"""6""",null,"""6""",null,null,"""53""",null
2,"""5102""",2,"""0""",17.5,"""1""",0.65,"""1""",3,null,"""53""",null
3,"""5405""",null,"""40""",null,"""1""",0.65,"""1""",3,null,"""53""",null
4,"""5405""",null,"""60""",null,"""6""",null,"""6""",null,null,"""53""",null
5,"""5405""",null,"""60""",null,"""1""",0.65,"""1""",3,null,"""53""",null
…,…,…,…,…,…,…,…,…,…,…,…
9,"""5405""",null,"""0""",19.5,"""1""",0.65,"""1""",3,null,"""53""",null
10,"""5102""",null,"""40""",null,"""1""",0.65,"""1""",3,null,"""53""",null
11,"""5405""",null,"""40""",null,"""6""",null,"""6""",null,null,"""53""",null
